# AI Labor Market Simulation - Complete Workflow

This notebook demonstrates the full simulation pipeline including:
1. Running a 20-year simulation
2. Collecting comprehensive metrics
3. Generating static plots
4. Creating interactive dashboards
5. Analyzing results

**Date:** April 2026  
**Duration:** ~5-10 minutes to run

## Step 1: Setup and Imports

Load all necessary modules and configure settings.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

# Suppress warnings for cleaner output``
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import simulation components
from src.simulation.engine import SimulationEngine
from src.config.parameters import DEFAULT_CONFIG
from src.analytics.plots import PlotGenerator, PlotConfig
from src.analytics.dashboard import DashboardBuilder, DashboardConfig
from src.simulation.benchmarks import BenchmarkRunner

print("✓ All imports successful")
print(f"Project root: {project_root}")

✓ All imports successful
Project root: c:\Users\mjohaadi\OneDrive - PineBridge Global Investments, LLC\Development\LAIM\ai_labor_market


## Step 2: Run Baseline Simulation

Execute a 20-year (240 period) simulation with default parameters.

In [2]:
print("Starting simulation...")
print("="*60)

# Initialize engine
config = DEFAULT_CONFIG.model_copy()
engine = SimulationEngine(config)

# Collect metrics during simulation
metrics_list = []
num_periods = 240  # 20 years

# Run simulation
for period in range(num_periods):
    engine.step()
    
    # Get statistics
    stats = engine.get_aggregate_statistics()
    stats['period'] = period
    metrics_list.append(stats)
    
    # Progress indicator
    if (period + 1) % 60 == 0:
        print(f"Completed {period + 1} periods ({(period + 1) / 12:.1f} years)")

# Create DataFrame
metrics_df = pd.DataFrame(metrics_list)

print("\n" + "="*60)
print(f"✓ Simulation complete!")
print(f"  Periods: {len(metrics_df)}")
print(f"  Years: {len(metrics_df) / 12:.1f}")
print(f"  Metrics columns: {len(metrics_df.columns)}")

Starting simulation...
Completed 60 periods (5.0 years)
Completed 120 periods (10.0 years)
Completed 180 periods (15.0 years)
Completed 240 periods (20.0 years)

✓ Simulation complete!
  Periods: 240
  Years: 20.0
  Metrics columns: 6


In [3]:
metrics_df.head()

,period,num_firms,unemployment_rate,market_wage_human,total_output,firms_in_market
0,0,3,0.050,0.999500,85.252455,"[0, 1, 2]"
1,1,0,0.065,0.997322,150.442889,[]
2,2,0,0.054,0.996425,0.000000,[]
3,3,0,0.078,0.993137,0.000000,[]
4,4,0,0.097,0.987972,0.000000,[]


## Step 3: Explore Metrics Data

View summary statistics from the simulation.

In [4]:
# Display first few rows
display("First 5 periods:")
display(metrics_df[['period', 'unemployment_rate', 'market_wage_human']].head())

print("\n" + "="*60)
print("Available metrics:")
print("="*60)
for i, col in enumerate(metrics_df.columns, 1):
    if col != 'period':
        print(f"{i:2d}. {col}")
        if i % 5 == 0:
            print()

'First 5 periods:'

,period,unemployment_rate,market_wage_human
0,0,0.050,0.999500
1,1,0.065,0.997322
2,2,0.054,0.996425
3,3,0.078,0.993137
4,4,0.097,0.987972



Available metrics:
 2. num_firms
 3. unemployment_rate
 4. market_wage_human
 5. total_output

 6. firms_in_market


## Step 4: Summary Statistics

View key statistics for the simulation run.

In [5]:
print("\nKEY SIMULATION STATISTICS")
print("="*60)

if 'unemployment_rate' in metrics_df.columns:
    unemp = metrics_df['unemployment_rate']
    print(f"\nUnemployment Rate:")
    print(f"  Initial: {unemp.iloc[0]:.2%}")
    print(f"  Final:   {unemp.iloc[-1]:.2%}")
    print(f"  Mean:    {unemp.mean():.2%}")
    print(f"  Range:   [{unemp.min():.2%}, {unemp.max():.2%}]")

if 'market_wage_h' in metrics_df.columns:
    wage = metrics_df['market_wage_h']
    growth = (wage.iloc[-1] / wage.iloc[0] - 1) * 100
    print(f"\nHuman Wage:")
    print(f"  Initial: ${wage.iloc[0]:,.0f}")
    print(f"  Final:   ${wage.iloc[-1]:,.0f}")
    print(f"  Growth:  {growth:+.1f}%")

# Count total employed
if 'total_employed_h' in metrics_df.columns:
    emp = metrics_df['total_employed_h']
    print(f"\nTotal Employed (Human):")
    print(f"  Initial: {emp.iloc[0]:,}")
    print(f"  Final:   {emp.iloc[-1]:,}")
    print(f"  Change:  {emp.iloc[-1] - emp.iloc[0]:+,}")

print("\n" + "="*60)


KEY SIMULATION STATISTICS

Unemployment Rate:
  Initial: 5.00%
  Final:   98.90%
  Mean:    79.71%
  Range:   [5.00%, 98.90%]



## Step 5: Generate Static Plots

Create publication-ready matplotlib/seaborn plots and save them.

In [6]:
print("Generating static plots...")
print("="*60)

# Create plot generator
plot_config = PlotConfig(
    figure_size=(14, 8),
    output_dir=Path(project_root) / 'outputs' / 'plots'
)
plot_gen = PlotGenerator(plot_config)

# Generate individual plots
print("\nGenerating plots:")
fig, ax = plot_gen.plot_unemployment_timeseries(metrics_df, save=True)
print("  ✓ Unemployment time series")

fig, ax = plot_gen.plot_wage_evolution(metrics_df, save=True)
print("  ✓ Wage evolution")

fig, ax = plot_gen.plot_ai_adoption(metrics_df, save=True)
print("  ✓ AI adoption")

fig, ax = plot_gen.plot_rd_spending(metrics_df, save=True)
print("  ✓ R&D spending")

print("\n✓ Static plots saved to outputs/plots/")

Generating static plots...

Generating plots:


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


  ✓ Unemployment time series


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


  ✓ Wage evolution


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


  ✓ AI adoption
  ✓ R&D spending

✓ Static plots saved to outputs/plots/


## Step 6: Create Interactive Dashboards

Build Plotly-based interactive dashboards for exploration.

In [7]:
print("Generating interactive dashboards...")
print("="*60)

# Create dashboard builder
dashboard_config = DashboardConfig(
    output_dir=Path(project_root) / 'outputs' / 'dashboards'
)
dashboard_builder = DashboardBuilder(dashboard_config)

# Generate all dashboards
dashboards = dashboard_builder.generate_all_dashboards(
    metrics_df,
    verbose=False
)

print(f"\n✓ Generated {len(dashboards)} interactive dashboards:")
print("\n" + "-"*60)
for name, path in sorted(dashboards.items()):
    print(f"  • {name:25s} → {path.name}")

print("\n" + "="*60)
print("\n💡 TIP: Open any .html file in your browser to interact!")

Generating interactive dashboards...

✓ Generated 5 interactive dashboards:

------------------------------------------------------------
  • firm_dynamics             → firm_dynamics_dashboard.html
  • inequality                → inequality_dashboard.html
  • labor_market              → labor_market_dashboard.html
  • overview                  → overview_dashboard.html
  • technology                → technology_dashboard.html


💡 TIP: Open any .html file in your browser to interact!


## Step 7: Run Benchmark Scenarios

Compare baseline with alternative scenarios.

In [2]:
print("\nRunning benchmark scenarios...")
print("="*60)

# Create benchmark runner
benchmark_runner = BenchmarkRunner(
    output_dir=Path(project_root) / 'outputs' / 'benchmarks',
    verbose=False
)

# Run a few quick scenarios for demonstration
scenarios = [
    BenchmarkRunner.BASELINE,
    BenchmarkRunner.HIGH_AI,
]

# Shorten for demo (normally 240 periods)
for scenario in scenarios:
    scenario.num_periods = 60  # Demo: 5 years instead of 20

print("\nRunning scenarios:")
for scenario in scenarios:
    print(f"  • {scenario.name}...", end="", flush=True)
    metrics, summary = benchmark_runner.run_scenario(scenario)
    print(" ✓")

# Compare scenarios
comparison = benchmark_runner.compare_scenarios(save=True)

print("\n" + "="*60)
print("\nScenario Comparison:")
print("-"*60)

# Display available columns and their values
if len(comparison) > 0:
    print(f"Available summary metrics: {list(comparison.columns)[:8]}...")
    print()
    # Show basic info that should be available
    for col in ['scenario_name', 'unemployment_mean', 'wage_growth_pct']:
        if col in comparison.columns:
            print(comparison[['scenario_name', col]].to_string(index=False))
            print()

print("✓ Results saved to outputs/benchmarks/")


Running benchmark scenarios...

Running scenarios:
  • Baseline... ✓
  • High AI... ✓


Scenario Comparison:
------------------------------------------------------------
Available summary metrics: ['scenario_name', 'num_periods', 'runtime_datetime', 'unemployment_mean', 'unemployment_std', 'unemployment_min', 'unemployment_max', 'wage_initial']...

scenario_name scenario_name
     Baseline      Baseline
      High AI       High AI

scenario_name  unemployment_mean
     Baseline           0.437483
      High AI           0.439067

scenario_name  wage_growth_pct
     Baseline       -91.057606
      High AI       -91.145258

✓ Results saved to outputs/benchmarks/


## Step 8: Detailed Analysis

Deep dive into specific aspects of the simulation.

In [7]:
print("\nDETAILED ANALYSIS")
print("="*60)

# Analyze unemployment dynamics
if 'unemployment_rate' in metrics.columns:
    print("\n1. UNEMPLOYMENT DYNAMICS")
    print("-"*60)
    unemp = metrics['unemployment_rate']
    
    # Period breakdown
    early = unemp.iloc[:60].mean()
    mid = unemp.iloc[60:180].mean()
    late = unemp.iloc[180:].mean()
    
    print(f"Early period (0-5 yrs):  {early:.2%}")
    print(f"Mid period (5-15 yrs):   {mid:.2%}")
    print(f"Late period (15-20 yrs): {late:.2%}")
    print(f"\nTrend: ", end="")
    if late < early:
        print(f"↓ Declining ({(early-late):.2%} improvement)")
    elif late > early:
        print(f"↑ Increasing ({(late-early):.2%} deterioration)")
    else:
        print("→ Stable")

# Analyze wage dynamics
if 'market_wage_h' in metrics.columns:
    print("\n2. WAGE DYNAMICS")
    print("-"*60)
    wage = metrics['market_wage_h']
    
    # Calculate quarterly growth rates
    quarterly_growth = wage.pct_change() * 100
    
    print(f"Average quarterly growth: {quarterly_growth.mean():+.3f}%")
    print(f"Volatility (std):         {quarterly_growth.std():.3f}%")
    print(f"Max increase:             {quarterly_growth.max():+.3f}% (period {quarterly_growth.idxmax()})")
    print(f"Max decrease:             {quarterly_growth.min():+.3f}% (period {quarterly_growth.idxmin()})")

# Analyze employment
if 'total_employed_h' in metrics.columns:
    print("\n3. EMPLOYMENT LEVELS")
    print("-"*60)
    emp = metrics['total_employed_h']
    labor_supply = 1000 / (1 - metrics['unemployment_rate'])  # Rough estimate
    employment_rate = emp / labor_supply if 'unemployment_rate' in metrics.columns else emp
    
    print(f"Initial employed:  {emp.iloc[0]:,} workers")
    print(f"Final employed:    {emp.iloc[-1]:,} workers")
    print(f"Net change:        {emp.iloc[-1] - emp.iloc[0]:+,} workers")
    
print("\n" + "="*60)


DETAILED ANALYSIS

1. UNEMPLOYMENT DYNAMICS
------------------------------------------------------------
Early period (0-5 yrs):  43.91%
Mid period (5-15 yrs):   nan%
Late period (15-20 yrs): nan%

Trend: → Stable



## Step 9: File Summary

Overview of all generated outputs.

In [8]:
from pathlib import Path
import os

print("\nGENERATED OUTPUT FILES")
print("="*60)

output_dir = Path(project_root) / 'outputs'

# Count files by category
categories = {
    'plots': output_dir / 'plots',
    'dashboards': output_dir / 'dashboards',
    'benchmarks': output_dir / 'benchmarks'
}

summary = {}
for category, path in categories.items():
    if path.exists():
        files = list(path.glob('**/*'))
        file_count = len([f for f in files if f.is_file()])
        summary[category] = file_count
        print(f"\n{category.upper()}:")
        print(f"  Location: {path}")
        print(f"  Files:    {file_count}")
        if file_count > 0:
            for f in sorted(files)[:5]:  # Show first 5
                if f.is_file():
                    size_mb = f.stat().st_size / (1024*1024)
                    print(f"    • {f.name} ({size_mb:.2f} MB)")
            if file_count > 5:
                print(f"    ... and {file_count - 5} more")

print("\n" + "="*60)
print("\n📊 SUMMARY")
print(f"Total files generated: {sum(summary.values())}")
print("\n✓ Simulation workflow complete!")


GENERATED OUTPUT FILES

PLOTS:
  Location: c:\Users\mjohaadi\OneDrive - PineBridge Global Investments, LLC\Development\LAIM\ai_labor_market\outputs\plots
  Files:    4
    • ai_adoption.png (0.09 MB)
    • rd_spending.png (0.09 MB)
    • unemployment_timeseries.png (0.14 MB)
    • wage_evolution.png (0.09 MB)

DASHBOARDS:
  Location: c:\Users\mjohaadi\OneDrive - PineBridge Global Investments, LLC\Development\LAIM\ai_labor_market\outputs\dashboards
  Files:    5
    • firm_dynamics_dashboard.html (4.63 MB)
    • inequality_dashboard.html (4.63 MB)
    • labor_market_dashboard.html (4.63 MB)
    • overview_dashboard.html (4.63 MB)
    • technology_dashboard.html (4.63 MB)

BENCHMARKS:
  Location: c:\Users\mjohaadi\OneDrive - PineBridge Global Investments, LLC\Development\LAIM\ai_labor_market\outputs\benchmarks
  Files:    5
    • metrics.csv (0.00 MB)
    • summary.csv (0.00 MB)
    • metrics.csv (0.00 MB)


📊 SUMMARY
Total files generated: 14

✓ Simulation workflow complete!


## Step 10: Next Steps

How to continue your analysis.

### 📁 Open Interactive Dashboards

1. Navigate to `outputs/dashboards/`
2. Open any `.html` file in your browser
3. Use interactive features:
   - **Hover**: See exact values
   - **Click-drag**: Pan/zoom
   - **Legend**: Toggle metrics

**Recommended order:**
```
1. overview_dashboard.html           (5-minute overview)
2. labor_market_dashboard.html       (unemployment & wages)
3. technology_dashboard.html         (AI & R&D dynamics)
4. scenario_comparison_*.html        (policy analysis)
```

### 📊 View Static Plots

High-quality PNG plots for presentations:
- `outputs/plots/unemployment_timeseries.png`
- `outputs/plots/wage_evolution.png`
- `outputs/plots/ai_adoption.png`
- `outputs/plots/rd_spending.png`

### 📈 Compare Scenarios

Find benchmark results in:
- `outputs/benchmarks/Baseline/metrics.csv`
- `outputs/benchmarks/High AI/metrics.csv`
- `outputs/benchmarks/scenario_comparison.csv`

### 🔧 Run Custom Analysis

Modify parameters in the cells above to:
- Change simulation length
- Adjust config parameters
- Run additional scenarios
- Generate custom plots

### 📚 Explore the Code

Key modules for deeper exploration:
- `src/simulation/engine.py` - Core simulation engine
- `src/analytics/dashboard.py` - Dashboard generation
- `src/analytics/plots.py` - Static plots
- `src/simulation/benchmarks.py` - Scenario definitions